# Telco Customer Churn EDA
Exploratory data analysis for the IBM Telco Customer Churn dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score


## Load dataset

In [ ]:
df = pd.read_csv('data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

## Check for missing values

In [ ]:
df.isnull().sum()

## Data Cleaning
- Partner, PaperlessBilling, and Churn converted to integers (1 for Yes, 0 for No).
- TotalCharges contained blanks for 11 customers with tenure 0 (less than a month); these rows were removed.

In [ ]:
yes_no_cols = ['Partner','PaperlessBilling','Churn']
le = LabelEncoder()
for col in yes_no_cols:
    df[col] = le.fit_transform(df[col])

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
rows_before = df.shape[0]
df = df.dropna(subset=['TotalCharges'])
rows_after = df.shape[0]
print(f'Dropped {rows_before-rows_after} rows due to missing TotalCharges')


## Summary statistics


In [ ]:
stats = df[['tenure','MonthlyCharges','TotalCharges']].agg(['count','mean','median','std','min','max'])
stats.loc['mode'] = df[['tenure','MonthlyCharges','TotalCharges']].mode().iloc[0]
stats


## Exploratory Data Analysis

### Gender and Churn Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].pie(df['gender'].value_counts(), labels=df['gender'].value_counts().index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Gender Distribution')
axes[1].pie(df['Churn'].value_counts(), labels=['No','Yes'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Churn Distribution')
plt.show()


### Customer contract distribution

In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(data=df, x='Contract', hue='Churn')
plt.title('Customer Contract Distribution by Churn')
plt.show()


### Payment method distribution

In [ ]:
order = df['PaymentMethod'].value_counts().index
fig, axes = plt.subplots(1, 2, figsize=(16,6))
sns.countplot(ax=axes[0], data=df, x='PaymentMethod', order=order)
axes[0].set_title('Payment Method Distribution')
axes[0].tick_params(axis='x', rotation=45)
ct = pd.crosstab(df['PaymentMethod'], df['Churn'])
ct.plot(kind='bar', stacked=True, ax=axes[1])
axes[1].set_title('Churn by Payment Method')
axes[1].set_xlabel('Payment Method')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Churn')
plt.tight_layout()
plt.show()


### Dependents and Partners

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,5))
sns.countplot(ax=axes[0], data=df, x='Dependents', hue='Churn')
axes[0].set_title('Dependents vs Churn')
sns.countplot(ax=axes[1], data=df, x='Partner', hue='Churn')
axes[1].set_title('Partner vs Churn')
plt.tight_layout()
plt.show()


### Services and Churn

In [ ]:
services = ['OnlineSecurity', 'PaperlessBilling', 'TechSupport', 'PhoneService']
fig, axes = plt.subplots(2, 2, figsize=(14,10))
for ax, service in zip(axes.flatten(), services):
    sns.countplot(ax=ax, data=df, x=service, hue='Churn')
    ax.set_title(f'Churn vs {service}')
plt.tight_layout()
plt.show()


### Charges and Tenure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18,5))
sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn', fill=True, ax=axes[0])
axes[0].set_title('Monthly Charges by Churn')
sns.kdeplot(data=df, x='TotalCharges', hue='Churn', fill=True, ax=axes[1])
axes[1].set_title('Total Charges by Churn')
sns.boxplot(x='Churn', y='tenure', data=df, ax=axes[2])
axes[2].set_title('Tenure vs Churn')
plt.tight_layout()
plt.show()


### Correlation Heatmap

In [ ]:
plt.figure(figsize=(25, 10))
corr = df.apply(lambda x: pd.factorize(x)[0]).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
ax = sns.heatmap(corr, mask=mask, xticklabels=corr.columns, yticklabels=corr.columns, annot=True, linewidths=.2, cmap='coolwarm', vmin=-1, vmax=1)
plt.show()

## Summary of useful plots

- **Crosstab / stacked bar**: categorical vs churn (gender, contract, payment, internet service).
- **Regplot / scatter**: numeric vs churn (monthly charges, tenure, total charges).
- **Box/violin plots**: numeric distributions by churn group.
- **Heatmaps**: churn proportions across service categories.

Below are example charts for each plot type.


### Crosstab / stacked bar example


In [ ]:
contract_churn = pd.crosstab(df['Contract'], df['Churn'])
contract_churn.div(contract_churn.sum(1), axis=0).plot(kind='bar', stacked=True)
plt.title('Contract Type vs Churn')
plt.ylabel('Proportion')
plt.show()


### Regplot examples for numeric features vs churn


In [ ]:
fig, axes = plt.subplots(1,3, figsize=(18,5))
sns.regplot(x='MonthlyCharges', y='Churn', data=df, ax=axes[0])
sns.regplot(x='tenure', y='Churn', data=df, ax=axes[1])
sns.regplot(x='TotalCharges', y='Churn', data=df, ax=axes[2])
axes[0].set_title('Monthly Charges')
axes[1].set_title('Tenure')
axes[2].set_title('Total Charges')
plt.tight_layout()
plt.show()


### Box plots by churn group


In [ ]:
fig, axes = plt.subplots(1,3, figsize=(18,5))
sns.boxplot(x='Churn', y='MonthlyCharges', data=df, ax=axes[0])
sns.boxplot(x='Churn', y='tenure', data=df, ax=axes[1])
sns.boxplot(x='Churn', y='TotalCharges', data=df, ax=axes[2])
axes[0].set_title('Monthly Charges')
axes[1].set_title('Tenure')
axes[2].set_title('Total Charges')
plt.tight_layout()
plt.show()


### Heatmap of service-related churn rates


In [ ]:
services = ['OnlineSecurity','TechSupport','PhoneService','InternetService']
heatmap_data = pd.DataFrame({feature: df.groupby(feature)['Churn'].mean() for feature in services}).T
sns.heatmap(heatmap_data, annot=True, cmap='Reds')
plt.title('Churn proportion by service category')
plt.ylabel('Service')
plt.xlabel('Category')
plt.show()


## Logistic Regression Model

In [ ]:
X = df.drop(columns=['Churn'])
X = pd.get_dummies(X, drop_first=True)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
preds = model.predict(X_test)
print('Accuracy:', accuracy_score(y_test, preds))
print(classification_report(y_test, preds))
